In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.utils.class_weight import compute_class_weight

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
df = pd.read_csv("test.csv", encoding="latin1")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (14402, 2)

Column names:
['text', 'sentiment']

Missing values:
text         0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
negative    9079
neutral     3046
positive    2277
Name: count, dtype: int64

First 5 rows:


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
stop_words = set(stopwords.words('english'))

# Keep sentiment-important negation words
negation_words = {'not', 'no', 'nor'}
stop_words = stop_words - negation_words


def preprocess(text):
    text = text.lower()

    tokens = word_tokenize(text)

    clean_tokens = [
        word
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    return ' '.join(clean_tokens)


df['clean_text'] = df['text'].apply(preprocess)

display(
    df[['text', 'clean_text']].head()
)

,text,clean_text
0,@VirginAmerica What @dhepburn said.,virginamerica dhepburn said
1,@VirginAmerica plus you've added commercials t...,virginamerica plus added commercials experienc...
2,@VirginAmerica I didn't today... Must mean I n...,virginamerica today must mean need take anothe...
3,@VirginAmerica it's really aggressive to blast...,virginamerica really aggressive blast obnoxiou...
4,@VirginAmerica and it's a really big bad thing...,virginamerica really big bad thing


In [ ]:
X = df['clean_text']
y = df['sentiment']

# First: separate the final test set (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second: split remaining 80% into 70% train and 10% validation
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.125,
    random_state=42,
    stratify=y_train_val
)

print("Training samples:", len(X_train_final))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train_final.value_counts())

print("\nValidation distribution:")
print(y_val.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training samples: 10080
Validation samples: 1441
Testing samples: 2881

Training distribution:
sentiment
negative    6355
neutral     2132
positive    1593
Name: count, dtype: int64

Validation distribution:
sentiment
negative    908
neutral     305
positive    228
Name: count, dtype: int64

Testing distribution:
sentiment
negative    1816
neutral      609
positive     456
Name: count, dtype: int64


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array(["negative", "neutral", "positive"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_final
)

class_weight_dict = dict(zip(classes, class_weights))#This converts the two arrays into a dictionary.

print("Class weights:")
print(class_weight_dict)

Class weights:
{np.str_('negative'): np.float64(0.5287175452399685), np.str_('neutral'): np.float64(1.575984990619137), np.str_('positive'): np.float64(2.109227871939736)}


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = vectorizer.fit_transform(X_train_final)

X_val_tfidf = vectorizer.transform(X_val)

X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Training TF-IDF shape: (10080, 5000)
Validation TF-IDF shape: (1441, 5000)
Testing TF-IDF shape: (2881, 5000)
Vocabulary size: 5000


In [ ]:
print("Training matrix shape:", X_train_tfidf.shape)
print("Number of non-zero values:", X_train_tfidf.nnz)

sparsity = 1 - (
    X_train_tfidf.nnz /
    (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
)

print("Sparsity:", sparsity)

print("\nFirst 10 vocabulary terms:")
print(vectorizer.get_feature_names_out()[:10])

Training matrix shape: (10080, 5000)
Number of non-zero values: 108221
Sparsity: 0.997852757936508

First 10 vocabulary terms:
['aa' 'aa flight' 'aa help' 'aa http' 'aa miles' 'aadvantage' 'aarp' 'abc'
 'ability' 'able']


In [ ]:
logistic_model = LogisticRegression(
    class_weight=class_weight_dict,
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_tfidf,
    y_train_final
)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [ ]:
y_val_pred = logistic_model.predict(X_val_tfidf)

print("Number of validation predictions:", len(y_val_pred))
print("Number of actual validation labels:", len(y_val))

Number of validation predictions: 1441
Number of actual validation labels: 1441


In [ ]:
val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

Validation Accuracy: 0.7863
Validation Accuracy: 78.63%


In [ ]:
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=['negative', 'neutral', 'positive']
    )
)

              precision    recall  f1-score   support

    negative       0.89      0.83      0.86       908
     neutral       0.59      0.66      0.62       305
    positive       0.69      0.79      0.73       228

    accuracy                           0.79      1441
   macro avg       0.72      0.76      0.74      1441
weighted avg       0.80      0.79      0.79      1441



In [ ]:
cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=['negative', 'neutral', 'positive']
)

print("Validation Confusion Matrix:")
print(cm)

Validation Confusion Matrix:
[[754 108  46]
 [ 69 200  36]
 [ 20  29 179]]


C controls the strength of regularization:

smaller C → stronger regularization → simpler model

larger C → weaker regularization → model can fit training data more closely

Remember: regularization is useful because our dataset is relatively small.

In [ ]:
C_values = [0.01, 0.1, 1, 10]

results = []

for C in C_values:

    model = LogisticRegression(
        C=C,
        max_iter=1000,
        class_weight=class_weight_dict,
        random_state=42
    )

    model.fit(
        X_train_tfidf,
        y_train_final
    )

    val_predictions = model.predict(X_val_tfidf)

    accuracy = accuracy_score(
        y_val,
        val_predictions
    )

    results.append({
        'C': C,
        'validation_accuracy': accuracy
    })

results_df = pd.DataFrame(results)

display(results_df)

,C,validation_accuracy
0,0.01,0.734906
1,0.10,0.752255
2,1.00,0.786260
3,10.00,0.771686


In [ ]:
final_logistic_model = LogisticRegression(
    C=1.0,
    class_weight=class_weight_dict,
    max_iter=1000,
    random_state=42
)

final_logistic_model.fit(
    X_train_tfidf,
    y_train_final
)

print("Final Logistic Regression model trained.")

Final Logistic Regression model trained.


In [ ]:
feature_names = vectorizer.get_feature_names_out()#name of features like good bad etc.

for class_index, class_name in enumerate(final_logistic_model.classes_):#final_logistic_model.classes_ it represent class 0,1,2

    coefficients = final_logistic_model.coef_[class_index]#coefficient for each class like for 0

    top_indices = coefficients.argsort()[-10:][::-1]#argsort do ascending so make it descending -10 for last 10 and -1 to reverse that, it gives coefficient index after sorting

    print(f"\nTop features for {class_name}:")

    for index in top_indices:
        print(
            f"{feature_names[index]:<20} "#<20 means space
            f"{coefficients[index]:.4f}"
        )


Top features for negative:
hours                3.2884
no                   3.1623
not                  3.1221
worst                2.9870
delayed              2.9380
hour                 2.7588
hold                 2.6993
luggage              2.4228
customers            2.2346
cancelled            2.1153

Top features for neutral:
hi                   1.9554
southwestair         1.9512
jetblue              1.7959
possible             1.7432
dm                   1.7401
tomorrow             1.6007
policy               1.5675
americanair          1.4349
chance               1.4340
united               1.4144

Top features for positive:
thank                5.6040
thanks               5.3923
great                4.5073
awesome              3.7891
love                 3.5776
amazing              3.2196
best                 3.1751
good                 2.7970
thx                  2.7444
appreciate           2.6989


In [ ]:
y_test_pred = final_logistic_model.predict(X_test_tfidf)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Accuracy: 0.7685
Test Accuracy: 76.85%


In [ ]:
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=['negative', 'neutral', 'positive']
    )
)

              precision    recall  f1-score   support

    negative       0.89      0.80      0.85      1816
     neutral       0.55      0.65      0.60       609
    positive       0.67      0.80      0.73       456

    accuracy                           0.77      2881
   macro avg       0.71      0.75      0.72      2881
weighted avg       0.79      0.77      0.77      2881



In [ ]:
cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=['negative', 'neutral', 'positive']
)

print("Final Logistic Regression Confusion Matrix:")
print(cm)

Final Logistic Regression Confusion Matrix:
[[1457  258  101]
 [ 137  393   79]
 [  34   58  364]]


In [ ]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average='macro'
)

weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average='weighted'
)

print("===== TF-IDF + Logistic Regression =====")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print("Best C        : 1.0")

===== TF-IDF + Logistic Regression =====
Test Accuracy : 76.85%
Macro F1      : 0.7235
Weighted F1   : 0.7746
Best C        : 1.0
